# ⚡ ValenWheel — Train the short-term forecaster on GPU (Google Colab)

Trains a **next-30-minute Valenbisi availability forecaster** on a large block of
15-minute snapshots, using the **T4 GPU**:

* **XGBoost** (`device="cuda"`) — served by the Streamlit app
* **PyTorch station-embedding MLP** — the deep-learning showcase

Both anchor on each station's *current* availability + its typical slot profile +
weather, and are scored against persistence & slot-mean baselines.

> **First: Runtime ▸ Change runtime type ▸ Hardware accelerator = T4 GPU.**
> Then Runtime ▸ Run all.

## 0 · Confirm the GPU is on

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')
!nvidia-smi -L

## 1 · Install deps (Colab already has torch + xgboost; pin sklearn)

In [ ]:
!pip -q install scikit-learn==1.7.2 xgboost==3.2.0 pyarrow

## 2 · Get the project code (edit the URL to your repo)

In [ ]:
REPO_URL = "https://github.com/jiale203/valenbisi.git"   # <-- your repo
import os
if not os.path.exists('valenbisi'):
    !git clone $REPO_URL valenbisi
%cd valenbisi

## 3 · Download a LARGE block of 15-minute data
More consecutive days → a better model (the GPU eats it easily). 120 days ≈ 3M rows.
Increase `--days` for an even stronger model (the repo covers 2022-12 → 2025-09).

In [ ]:
!python download_forecast_data.py --start 2025-01-01 --days 120

## 4 · Train both models on the GPU
`train_forecast.py` auto-detects CUDA: XGBoost runs on GPU and the PyTorch net trains on GPU.

In [ ]:
!python train_forecast.py --epochs 40

## 5 · Results — model vs baselines

In [ ]:
import json, pandas as pd
m = json.load(open('models/forecast_meta.json'))
print('Horizon:', m['horizon_min'], 'min | rows:', f"{m['n_rows']:,}", '| device:', m['device'])
rows = [{'model':'persistence', **m['baselines']['persistence']},
        {'model':'slot_mean',   **m['baselines']['slot_mean']},
        {'model':'XGBoost',     **m['xgboost']}]
if m.get('net'): rows.append({'model':'PyTorch net', **m['net']})
pd.DataFrame(rows)

In [ ]:
# XGBoost feature importance
import joblib, pandas as pd
b = joblib.load('models/forecast_short.pkl')
imp = pd.Series(b['model'].feature_importances_, index=b['features']).sort_values()
imp.plot.barh(figsize=(6,5), title='XGBoost feature importance'); 

## 6 · Download the trained artifacts
Drop `models/forecast_*` and `data/slot_profile.parquet` back into your repo and redeploy.

In [ ]:
import shutil, os
os.makedirs('out', exist_ok=True)
for f in ['models/forecast_short.pkl','models/forecast_net.pt',
          'models/forecast_net_cfg.json','models/forecast_meta.json',
          'data/slot_profile.parquet']:
    shutil.copy(f, 'out/')
shutil.make_archive('valenwheel_forecast', 'zip', 'out')
try:
    from google.colab import files; files.download('valenwheel_forecast.zip')
except Exception as e:
    print('Not in Colab; see valenwheel_forecast.zip', e)